In [ ]:
import numpy as np
from scipy import signal
from matplotlib import pyplot as plt

## Filter functions used in the phase transition detection
This notebook requires review, many functions defined here are now part of the library. 

In [ ]:
SOBEL_X = np.array([[1, 0, -1],
                    [2, 0, -2],
                    [1, 0, -1]])
SOBEL = SOBEL_X + SOBEL_X.T * 1j


def bump(x, *, _sqrt=False):
    # Implement the function exp(-1/(1-x^2)) for x in (-1, 1) and
    # 0 for |x|>=1.
    x = np.where(np.abs(x) >= 1, 1 - np.finfo(float).eps, x)
    x = np.exp(-1/(1 - x)) if _sqrt else np.exp(-1/(1 - x**2))
    return x


def bump_rbf(x):
    # Implement the function bump(|x|_2) where |.|_2 if the 2 norm.
    # Input is a tensor where the last dimension is the vector for the bump.
    x = np.sum(np.square(x), axis=-1)
    return bump(x, _sqrt=True)


def bump_rbf_2d_dx(v):
    # Derivative wrt x of the bump rbf function.
    # s = -2x / (1-x^2-y^2)^2
    s = np.sum(np.square(v), axis=-1)
    s = np.square(1 - s)
    s = - 2 * v[:,0] / s
    return bump_rbf(v) * s


def bump_lattice_xy(n):
    xy = np.linspace(-1, 1, n), np.linspace(-1, 1, n)
    xy = map(lambda m: m.flatten(), np.meshgrid(*xy, indexing='xy'))
    xy = tuple(xy)
    xy = np.stack(xy).T
    return xy


def bump_kernel(n, *, scale=1.):
    # Prepare the bump kernel with n steps.
    xy = bump_lattice_xy(n)
    kernel = bump_rbf(xy)
    kernel = np.reshape(kernel, (n, n))
    kernel = scale * kernel / np.sum(kernel)

    return kernel


def grad_bump_kernel(n):
    xy = bump_lattice_xy(n)
    kernel_x = bump_rbf_2d_dx(xy)
    kernel_x = np.reshape(kernel_x, (n, n))
    kernel_y = bump_rbf_2d_dx(xy[:, ::-1])  # Reverse order of x, y to obtain dy.
    kernel_y = np.reshape(kernel_y, (n, n))
    return kernel_x, kernel_y

In [ ]:
# This is the derivative obtained using Sobel.

kernel = bump_kernel(64, scale=1)
kernel = signal.convolve2d(kernel, SOBEL, boundary='symm', mode='same')

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].set_title('$\\mathfrak{Re}(k)$')
axs[0].matshow(np.real(kernel), cmap='hot', aspect='equal')
axs[1].set_title('$\\mathfrak{Im}(k)$')
imgax = axs[1].matshow(np.imag(kernel), cmap='hot', aspect='equal')
plt.colorbar(imgax)
plt.tight_layout()

- Bump function: $b(x)=e^{-\frac{1}{1-x^2}}$ for $x \in (-1, 1)$ and $b(x)=0$ for $|x|\ge 1$.
- Bump RBF 2D: $k(x, y)=e^{-\frac{1}{1-x^2-y^2}}$ for $\|(x, y)\|_2 \in [0, 1)$ and 0 otherwise.
- Bump RBF dx: $k_x(x, y)=-\frac{2x}{(1-x^2-y^2)^2} k(x, y)$. Also dy: $k_y(x, y)=k_x(y, x)$.

In [ ]:
# This is the derivative obtained directly (discretization of the analytic form).
# Note this is not normalized such that the integral over the entire domain of the bump rbf (before the derivative) is 1.
kernel_x, kernel_y = grad_bump_kernel(64)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].set_title('$\\mathfrak{Re}(k)$')
axs[0].matshow(kernel_x, cmap='hot', aspect='equal')
axs[1].set_title('$\\mathfrak{Im}(k)$')
imgax = axs[1].matshow(kernel_y, cmap='hot', aspect='equal')
plt.colorbar(imgax)
plt.tight_layout()

Assume we have a scalar function $g: \mathbb{R}^2 \to \mathbb{R}$. We sample $g$ on a regular grid (on a subset of the domain $\mathbb{R}^2$) lattice to obtain an approximation of the convolution $g\star k$ where $k$ is the kernel of the convolution.

In [ ]:
# Some silly function g.
g = np.full((64, 64), -1)
g[:32, :32] = 1
g[32:64, 32:64] = 1
g = g * np.linspace(0, 1, 64)
g = g + np.random.normal(size=(64, 64)) * 0.01

# Prepare the convolutions for the two smoothed components of the gradient.
kernel_x, kernel_y = grad_bump_kernel(6)    # The filter width determines the smoothness.
# Convolve g with the kernel above.
grad_g_x = signal.convolve2d(g, kernel_x, boundary='symm', mode='same')
grad_g_y = signal.convolve2d(g, kernel_y, boundary='symm', mode='same')

# Coordinates x, y of the point a in grid coordinates (for simplicity) used
# in the next cell.
a = (45, 32)

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].matshow(g, cmap='hot', origin='lower')
axs[0].plot([a[0]], [a[1]], marker='x', color='red')
axs[0].set_title('Function $g$')
axs[1].matshow(grad_g_x, cmap='hot', origin='lower')
axs[1].set_title('Conv $g \\star k_x$')
imgax = axs[2].matshow(grad_g_y, cmap='hot', origin='lower')
axs[2].set_title('Conv $g \\star k_y$')
plt.colorbar(imgax)
plt.tight_layout()

In [ ]:
# Angle of vector field for various rotations of the colormap.
cplx_grad_g = signal.convolve2d(g, kernel_x + 1j * kernel_y, boundary='symm', mode='same')
fig, axs = plt.subplots(1, 4, figsize=(10, 4))
for i, c in enumerate((1, -1, 1j, -1j)):
    axs[i].matshow(np.angle(cplx_grad_g * c), cmap='twilight')
    axs[i].set_xticks([])
    axs[i].set_yticks([])

In [ ]:
# Obtain integral of g(x-a) * kernel_x, where * is the multiplication of functions.
# The point 'a' is marked in the plot of g above.

khsz = len(kernel_x) // 2    # kernel must have an even number of rows/cols.
g_kernel = g[a[1] - khsz: a[1] + khsz, a[0] - khsz: a[0] + khsz]    # Extract region of g aounrd a.
# Compute the two (discretized) integrals.
np.sum(g_kernel * kernel_x), np.sum(g_kernel * kernel_y)

In [ ]:
# Details of the components for the discretized integrals used in the previous cell.

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].matshow(g_kernel, cmap='hot', origin='lower')
axs[0].set_title('Neighbor of $g$ at $a$')
axs[1].set_title('Kernel x')
axs[1].matshow(kernel_x, cmap='hot', origin='lower')
imgax = axs[2].matshow(kernel_y, cmap='hot', origin='lower')
axs[2].set_title('Kernel y')
plt.colorbar(imgax)
plt.tight_layout()